<a href="https://colab.research.google.com/github/Bachbean/Predicting-the-Unpredictable/blob/main/scripts/CM1_surrogate_prediction_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
import requests
from io import StringIO
from scipy.spatial import cKDTree


# ============================================================
# 1. LOAD CSV FROM GITHUB
# ============================================================

url = "https://raw.githubusercontent.com/Bachbean/Predicting-the-Unpredictable/refs/heads/main/scripts/CM1_all_vars_30s.csv"

response = requests.get(url)

response.raise_for_status()

raw_text = response.text


# ============================================================
# 2. READ CSV AND IGNORE FIRST LINE
# ============================================================

data_all = np.loadtxt(
    StringIO(raw_text),
    delimiter=",",
    dtype=np.float64,
    skiprows=1
)

data = data_all[:10000]


if data.ndim != 2 or data.shape[1] != 4:

    raise ValueError(
        "Expected exactly four comma-separated numerical columns."
    )


# ============================================================
# 3. SPLIT INTO FOUR ARRAYS
# ============================================================

data1 = data[:, 0]
data2 = data[:, 1]
data3 = data[:, 2]
data4 = data[:, 3]


all_data = [
    data1,
    data2,
    data3,
    data4
]


# ============================================================
# 4. ENTER RESULTS FROM AMI AND FNN
# ============================================================

# Replace these with your actual tau values

tau_by_column = [
    112,
    69,
    67,
    78
]


# Replace these with your actual embedding dimensions

m_by_column = [
    4,
    4,
    4,
    4
]


# ============================================================
# 5. SETTINGS
# ============================================================

k_neighbors = 10


prediction_horizons = [
    1,
    5,
    10,
    20,
    40
]


n_prediction_points = 2000


n_surrogates = 99


rng = np.random.default_rng(
    12345
)


# ============================================================
# 6. DELAY EMBEDDING FUNCTION
# ============================================================

def delay_embed(
    series,
    m,
    tau
):

    N = (
        len(series)
        -
        (m - 1) * tau
    )


    if N <= 0:

        raise ValueError(
            "Embedding parameters are too large."
        )


    X = np.empty(
        (N, m)
    )


    for j in range(m):

        X[:, j] = series[
            j * tau:
            j * tau + N
        ]


    return X


# ============================================================
# 7. PHASE-RANDOMIZED SURROGATE FUNCTION
# ============================================================

def phase_randomized_surrogate(
    series,
    rng
):

    N = len(series)


    spectrum = np.fft.rfft(
        series
    )


    surrogate_spectrum = (
        spectrum.copy()
    )


    if N % 2 == 0:

        phase_indices = np.arange(
            1,
            len(spectrum) - 1
        )

    else:

        phase_indices = np.arange(
            1,
            len(spectrum)
        )


    random_phases = rng.uniform(
        0,
        2 * np.pi,
        len(phase_indices)
    )


    surrogate_spectrum[
        phase_indices
    ] = (

        np.abs(
            spectrum[
                phase_indices
            ]
        )

        *

        np.exp(
            1j * random_phases
        )
    )


    # Preserve DC component exactly

    surrogate_spectrum[0] = (
        spectrum[0]
    )


    # Preserve Nyquist component exactly

    if N % 2 == 0:

        surrogate_spectrum[-1] = (
            spectrum[-1]
        )


    surrogate = np.fft.irfft(
        surrogate_spectrum,
        n=N
    )


    return surrogate


# ============================================================
# 8. NONLINEAR LOCAL PREDICTION ERROR FUNCTION
# ============================================================

def nonlinear_prediction_error(
    series,
    tau,
    m,
    theiler_window,
    k_neighbors,
    horizons,
    n_prediction_points=2000,
    query_seed=2026
):

    max_horizon = max(
        horizons
    )


    X = delay_embed(
        series,
        m,
        tau
    )


    state_times = (

        np.arange(
            len(X)
        )

        +

        (m - 1) * tau

    )


    usable_points = (
        len(X)
        -
        max_horizon
    )


    X_search = X[
        :usable_points
    ]


    state_times = state_times[
        :usable_points
    ]


    tree = cKDTree(
        X_search
    )


    query_rng = (
        np.random.default_rng(
            query_seed
        )
    )


    if (
        n_prediction_points
        >=
        usable_points
    ):

        query_indices = np.arange(
            usable_points
        )

    else:

        query_indices = np.sort(

            query_rng.choice(

                usable_points,

                size=n_prediction_points,

                replace=False
            )

        )


    number_candidates = min(

        max(

            50,

            4 * k_neighbors,

            2 * theiler_window + 20

        ),

        usable_points

    )


    distances, indices = tree.query(

        X_search[
            query_indices
        ],

        k=number_candidates

    )


    squared_errors = {

        h: []

        for h in horizons

    }


    for row, i in enumerate(
        query_indices
    ):

        neighbors = []


        for candidate in np.atleast_1d(
            indices[row]
        ):

            if candidate >= usable_points:
                continue


            if (

                abs(

                    state_times[candidate]
                    -
                    state_times[i]

                )

                <=
                theiler_window

            ):

                continue


            neighbors.append(
                candidate
            )


            if (
                len(neighbors)
                >=
                k_neighbors
            ):

                break


        if (
            len(neighbors)
            <
            k_neighbors
        ):

            continue


        neighbors = np.asarray(
            neighbors,
            dtype=int
        )


        for h in horizons:

            neighbor_future_values = series[

                state_times[
                    neighbors
                ]

                +

                h
            ]


            prediction = np.mean(
                neighbor_future_values
            )


            actual = series[

                state_times[i]

                +

                h
            ]


            squared_errors[h].append(

                (
                    prediction
                    -
                    actual
                ) ** 2

            )


    scale = np.std(
        series
    )


    if scale == 0:

        raise ValueError(
            "Series has zero variance."
        )


    nrmse = {}


    for h in horizons:

        errors = np.asarray(
            squared_errors[h]
        )


        if len(errors) == 0:

            nrmse[h] = np.nan

        else:

            rmse = np.sqrt(
                np.mean(errors)
            )


            nrmse[h] = (
                rmse
                /
                scale
            )


    valid_values = np.array(

        [
            value

            for value
            in nrmse.values()

            if np.isfinite(value)
        ]

    )


    prediction_score = np.mean(
        valid_values
    )


    return (
        prediction_score,
        nrmse
    )


# ============================================================
# 9. RUN SURROGATE TEST FOR ALL FOUR COLUMNS
# ============================================================

results = []


for column in range(4):

    print(
        "\n\n======================================"
    )

    print(
        f"COLUMN {column + 1}"
    )

    print(
        "======================================"
    )


    series = all_data[column]


    tau = tau_by_column[column]

    m = m_by_column[column]


    # Use tau as initial Theiler window

    theiler_window = tau


    # --------------------------------------------------------
    # REAL DATA PREDICTION SCORE
    # --------------------------------------------------------

    real_score, real_nrmse = (
        nonlinear_prediction_error(

            series=series,

            tau=tau,

            m=m,

            theiler_window=theiler_window,

            k_neighbors=k_neighbors,

            horizons=prediction_horizons,

            n_prediction_points=n_prediction_points

        )
    )


    # --------------------------------------------------------
    # SURROGATES
    # --------------------------------------------------------

    surrogate_scores = []


    for i in range(
        n_surrogates
    ):

        surrogate = (
            phase_randomized_surrogate(
                series,
                rng
            )
        )


        score, _ = (
            nonlinear_prediction_error(

                series=surrogate,

                tau=tau,

                m=m,

                theiler_window=theiler_window,

                k_neighbors=k_neighbors,

                horizons=prediction_horizons,

                n_prediction_points=n_prediction_points

            )
        )


        surrogate_scores.append(
            score
        )


    surrogate_scores = np.asarray(
        surrogate_scores
    )


    valid = np.isfinite(
        surrogate_scores
    )


    surrogate_scores = (
        surrogate_scores[valid]
    )


    # --------------------------------------------------------
    # EMPIRICAL P-VALUE
    #
    # LOWER prediction score = better prediction
    # --------------------------------------------------------

    number_as_good_or_better = np.sum(

        surrogate_scores
        <=
        real_score

    )


    p_value = (

        number_as_good_or_better
        +
        1

    ) / (

        len(surrogate_scores)
        +
        1

    )


    percentile = (

        np.sum(

            surrogate_scores
            >
            real_score

        )

        /

        len(surrogate_scores)

    ) * 100


    # --------------------------------------------------------
    # SAVE RESULT
    # --------------------------------------------------------

    results.append({

        "tau": tau,

        "m": m,

        "real_prediction_score":
            real_score,

        "mean_surrogate_score":
            np.mean(
                surrogate_scores
            ),

        "median_surrogate_score":
            np.median(
                surrogate_scores
            ),

        "minimum_surrogate_score":
            np.min(
                surrogate_scores
            ),

        "maximum_surrogate_score":
            np.max(
                surrogate_scores
            ),

        "p_value":
            p_value,

        "percentile":
            percentile

    })


    # --------------------------------------------------------
    # PRINT RESULT
    # --------------------------------------------------------

    print(
        f"tau = {tau}"
    )

    print(
        f"m = {m}"
    )


    print(
        "\nPrediction error by horizon:"
    )


    for h in prediction_horizons:

        print(

            f"Horizon {h:3d}: "
            f"{real_nrmse[h]:.6f}"

        )


    print(
        f"\nOriginal prediction score = "
        f"{real_score:.6f}"
    )


    print(
        f"Mean surrogate prediction score = "
        f"{np.mean(surrogate_scores):.6f}"
    )


    print(
        f"Median surrogate prediction score = "
        f"{np.median(surrogate_scores):.6f}"
    )


    print(
        f"Empirical p-value = "
        f"{p_value:.4f}"
    )


    print(
        f"Original data predicts better than "
        f"{percentile:.2f}% "
        f"of surrogates."
    )


# ============================================================
# 10. FINAL SUMMARY
# ============================================================

print(
    "\n\n======================================"
)

print(
    "FINAL SUMMARY"
)

print(
    "======================================"
)


for column in range(4):

    result = results[column]


    print(
        f"\nColumn {column + 1}:"
    )


    print(
        f"  tau = "
        f"{result['tau']}"
    )


    print(
        f"  m = "
        f"{result['m']}"
    )


    print(
        f"  real prediction score = "
        f"{result['real_prediction_score']:.6f}"
    )


    print(
        f"  mean surrogate score = "
        f"{result['mean_surrogate_score']:.6f}"
    )


    print(
        f"  p-value = "
        f"{result['p_value']:.4f}"
    )


    print(
        f"  better than "
        f"{result['percentile']:.2f}% "
        f"of surrogates"
    )



COLUMN 1
tau = 112
m = 4

Prediction error by horizon:
Horizon   1: 0.091874
Horizon   5: 0.095726
Horizon  10: 0.104766
Horizon  20: 0.128836
Horizon  40: 0.169738

Original prediction score = 0.118188
Mean surrogate prediction score = 0.242521
Median surrogate prediction score = 0.241248
Empirical p-value = 0.0100
Original data predicts better than 100.00% of surrogates.


COLUMN 2
tau = 69
m = 4

Prediction error by horizon:
Horizon   1: 0.140886
Horizon   5: 0.173081
Horizon  10: 0.231388
Horizon  20: 0.327633
Horizon  40: 0.404309

Original prediction score = 0.255459
Mean surrogate prediction score = 0.312242
Median surrogate prediction score = 0.311075
Empirical p-value = 0.0100
Original data predicts better than 100.00% of surrogates.


COLUMN 3
tau = 67
m = 4

Prediction error by horizon:
Horizon   1: 0.168681
Horizon   5: 0.300331
Horizon  10: 0.482352
Horizon  20: 0.745820
Horizon  40: 0.938380

Original prediction score = 0.527113
Mean surrogate prediction score = 0.57262